# 01 — 数据探索与可视化

本 Notebook 展示 HUSTmotor 数据集的加载、波形可视化和基本统计分析。
运行前请确保已安装依赖：`pip install -r requirements.txt`

In [ ]:
import os, sys
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from src.config import DATA_DIR, SKIPROWS, DATA_COLUMNS, SIGNAL_COLUMNS, FILE_ENCODING, SEPARATOR, LABEL_MAP

%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['figure.dpi'] = 100

In [ ]:
# 列出数据集中的所有文件
data_files = sorted(Path(DATA_DIR).glob('*.txt'))
print(f'共找到 {len(data_files)} 个数据文件:\n')
for f in data_files:
    size_mb = f.stat().st_size / (1024 * 1024)
    print(f'  {f.name:<30s} {size_mb:6.1f} MB')

In [ ]:
# 解析文件名中的标签和转速
import re
for fp in data_files[:6]:
    stem = os.path.splitext(fp.name)[0]
    match = re.match(r'([A-Za-z]+)_(\d+)HZ', stem, re.IGNORECASE)
    raw_label = match.group(1).upper()
    speed = int(match.group(2))
    label = LABEL_MAP.get(raw_label, raw_label)
    print(f'  {fp.name:<30s} → 标签={label:<4s}  转速={speed} Hz')

In [ ]:
# 加载一个文件并查看数据形状
fp = str(data_files[0])
df = pd.read_csv(fp, skiprows=SKIPROWS, sep=SEPARATOR, header=None, names=DATA_COLUMNS, encoding=FILE_ENCODING, engine='python')

print(f'文件: {data_files[0].name}')
print(f'数据量: {len(df)} 行 × {len(df.columns)} 列')
print(f'采样率: 25600 Hz, 时长: {len(df)/25600:.2f} 秒')
print(f'\n数据统计摘要:')
display(df.describe())

In [ ]:
# 绘制四通道波形（前 2048 个采样点）
signal_data = df[SIGNAL_COLUMNS].values[:2048].astype(np.float64)
time = np.arange(2048) / 25600 * 1000  # 转为毫秒

fig, axes = plt.subplots(4, 1, figsize=(14, 12), sharex=True)
channels = ['X (振动)', 'Y (振动)', 'Z (振动)', 'Sound (声学)']
colors = ['#00e5ff', '#ff6d00', '#76ff03', '#e040fb']

for i, (ax, ch, c) in enumerate(zip(axes, channels, colors)):
    ax.plot(time, signal_data[:, i], color=c, linewidth=0.6)
    ax.set_ylabel(ch, color=c, fontsize=12)
    ax.grid(True, alpha=0.2)
    ax.set_facecolor('#0a0e27')

axes[-1].set_xlabel('时间 (ms)', fontsize=12)
fig.patch.set_facecolor('#0a0e27')
fig.suptitle(f'四通道原始波形 — {data_files[0].name}', color='white', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# 对比不同转速下的波形（X通道）
speed_files = {}
for f in data_files:
    if 'H_' in f.name:  # 只看健康状态
        speed = int(re.match(r'H_(\d+)HZ', f.name).group(1))
        speed_files[speed] = str(f)

fig, axes = plt.subplots(4, 1, figsize=(14, 12), sharex=True)
fig.patch.set_facecolor('#0a0e27')

for ax, (speed, fp) in zip(axes, sorted(speed_files.items())):
    df = pd.read_csv(fp, skiprows=SKIPROWS, sep=SEPARATOR, header=None, names=DATA_COLUMNS, encoding=FILE_ENCODING, engine='python')
    signal = df['X'].values[:2048].astype(np.float64)
    t = np.arange(2048) / 25600 * 1000
    ax.plot(t, signal, color='#00e5ff', linewidth=0.6)
    ax.set_ylabel(f'{speed} Hz', color='#7eb8da', fontsize=12)
    ax.grid(True, alpha=0.2)
    ax.set_facecolor('#0a0e27')

axes[-1].set_xlabel('时间 (ms)', fontsize=12, color='white')
fig.suptitle('不同转速下 X 通道波形对比 (健康状态)', color='white', fontsize=14)
plt.tight_layout()
plt.show()

### 关键观察

1. 相同故障在不同转速下的振动幅值差异可达 6-7 倍 → 需要无量纲特征消除幅值影响
2. 采样率 25.6 kHz，每个文件约 6.4 秒数据，足够覆盖多周期振动
3. Sound 通道的高频成分明显多于振动通道，对轴承故障诊断有价值